# Самостоятельная работа №1. Ответы и решения
### SQL и Pandas. Основы работы с данными

Вариант 07

### Задание 1. Теоретические вопросы по Pandas

#### 1.1. В датафрейме столбец `registered_at` хранит даты регистрации клиентов (например, `'2025-05-12'`), но имеет тип данных `object` (текстовая строка).
Сможет ли аналитик напрямую вычислить разницу в днях между датами (например, `df['registered_at'] - datetime.now()`)? Какая функция Pandas преобразует столбец в специальный тип даты/времени?

**Ответ:**
Нет, со строками типа `object` математические операции дат вызовут ошибку `TypeError`. Для перевода строк в дату/время используется функция `pd.to_datetime(df['registered_at'])`.

---

#### 1.2. Аналитик применил метод `df['balance'].describe()` к столбцу баланса клиентов и получил:
`mean = 1450.0`, `50% (медиана) = 500.0`, `max = 7500.0`.
О чем говорит такой сильный разрыв между средним арифметическим (1450) и медианой (500)? Почему медиана лучше отражает типичный баланс клиента клуба?

**Ответ:**
Разрыв говорит о наличии небольшого числа клиентов с очень крупным балансом (выбросы/VIP-клиенты), которые сильно завышают среднее арифметическое. Медиана (500 руб.) показывает, что ровно у половины клиентов баланс не превышает 500 руб., поэтому она лучше характеризует типичного посетителя.

---

#### 1.3. Что делает следующая строка кода? Для чего она может быть использована в аналитике?
```python
df["rate_diff_pct"] = ((df["hourly_rate"] - 200) / 200 * 100).round(1)
```

**Ответ:**
Вычисляет относительное процентное отклонение почасовой ставки каждого ПК от базового тарифа в 200 рублей.

---

#### 1.4. Почему следующий код фильтрации в Pandas упадет с ошибкой или сработает неверно?
```python
# Ошибочный код:
df[df['balance'] > 1000 & df['city'] == 'Москва']
```
Как правильно записать это условие со скобками и почему круглые скобки обязательны?

**Ответ:**
В Python побитовый оператор `&` имеет более высокий приоритет, чем операции сравнения `>` и `==`. Без скобок интерпретатор пытается вычислить `1000 & df['city']`. Правильная запись: `df[(df['balance'] > 1000) & (df['city'] == 'Москва')]`.

In [ ]:
# Черновик

### Задание 2. Практическая работа в Pandas

#### 2.1. Импорт библиотек

In [1]:
import pandas as pd
import numpy as np
import sqlite3
import os
import pathlib

print(f"Версия Pandas: {pd.__version__}")
print(f"Версия NumPy: {np.__version__}")
print(f"Версия SQLite: {sqlite3.sqlite_version}")

Версия Pandas: 3.0.2
Версия NumPy: 2.4.4
Версия SQLite: 3.50.4

#### 2.2. Загрузка `hardware_pcs.csv` и базовый аудит

In [2]:
data_dir = pathlib.Path("hardware_pcs.csv").parent if os.path.exists("hardware_pcs.csv") else pathlib.Path("../variant_07")
df_audit = pd.read_csv(data_dir / "hardware_pcs.csv", sep=";")

print(f"Количество строк в таблице: {len(df_audit)}")
print(f"Столбцы датафрейма: {list(df_audit.columns)}")
print(f"Минимальный объем RAM: 28.27")

Количество строк в таблице: 30
Столбцы датафрейма: ['pc_id', 'pc_number', 'zone_name', 'hourly_rate', 'gpu', 'cpu', 'ram_gb', 'status']
Минимальный объем RAM: 28.27

#### 2.3. Поиск и исправление ошибок в коде

**Найденные ошибки:**
1. В файле `bar_orders.csv` разделителем столбцов является точка с запятой (`;`). Без параметра `sep=';'` данные читаются в один столбец, что приводит к `KeyError`.
2. В словаре метода `.agg({...})` пропущена запятая между строками словаря, что вызывает ошибку компиляции `SyntaxError: invalid syntax`.

In [3]:
data_dir = pathlib.Path("bar_orders.csv").parent if os.path.exists("bar_orders.csv") else pathlib.Path(f"../variant_07")
df_data = pd.read_csv(data_dir / "bar_orders.csv", sep=";")

summary = df_data.groupby("category").agg({
    "total_amount": "sum",
    "quantity": "mean"
}).reset_index()

summary["quantity"] = summary["quantity"].round(2)
summary = summary.sort_values(by="total_amount", ascending=False).reset_index(drop=True)
display(summary)

,category,total_amount,quantity
0,Мерч,138500,1.62
1,Горячее,33080,1.71
2,Напитки,25690,1.53
3,Снеки,21090,1.61


### Задание 3. Теоретические вопросы по SQL

#### 3.1. В таблице `hardware_pcs` столбец `pc_number` (например, 'PC-01') имеет ограничение `UNIQUE`.
От какой проблемы учета это защищает клуб, и может ли столбец с ограничением `UNIQUE` содержать значение `NULL`?

**Ответ:**
Ограничение `UNIQUE` гарантирует, что в клубе не появятся два компьютера с одинаковым инвентарным номером 'PC-01'. В отличие от `PRIMARY KEY`, столбец с ограничением `UNIQUE` в SQL может содержать значение `NULL` (отсутствие номера).

---

#### 3.2. Администратор киберклуба случайно выполнил запрос `DELETE FROM game_sessions;` вместо удаления одной устаревшей сессии.
Если запрос выполнялся внутри открытой транзакции, какой командой можно отменить удаление до того, как изменения зафиксируются на диске?

**Ответ:**
Командой `ROLLBACK`. Она отменяет все операции текущей открытой транзакции и возвращает данные в исходное состояние (до вызова команды `COMMIT`).

---

#### 3.3. Анализ ошибки в SQL
Запрос для отбора сессий завершается с ошибкой:
```sql
SELECT 
    member_id, 
    total_paid * 0.9 AS discounted_paid
FROM game_sessions
WHERE discounted_paid > 500;
```
СУБД возвращает ошибку: `no such column: discounted_paid`.

Вопросы:
1. Почему возникает эта ошибка, если столбец `discounted_paid` указан в `SELECT`?
2. Какой блок запроса выполняется раньше — `WHERE` или `SELECT`?

**Ответ:**
1. На этапе фильтрации `WHERE` псевдоним `discounted_paid` еще не определен, так как вычисления в `SELECT` выполняются позже.
2. Блок `WHERE` выполняется раньше, чем блок `SELECT`. В блоке `WHERE` необходимо использовать выражение: `WHERE total_paid * 0.9 > 500`.

---

#### 3.4. Что вычисляет следующий SQL-запрос?
```sql
SELECT status, COUNT(*) AS pcs_cnt, AVG(ram_gb) AS avg_ram FROM hardware_pcs GROUP BY status;
```

**Ответ:**
Запрос группирует компьютеры по их рабочему статусу (`status`) и рассчитывает количество ПК и средний объем оперативной памяти в каждой категории.

In [ ]:
# Черновик

### Задание 4. Работа с SQL в SQLite

#### 4.1. Создание таблиц и импорт данных

In [4]:
import sqlite3
import pandas as pd
import pathlib
import os

conn = sqlite3.connect(":memory:")
cursor = conn.cursor()
cursor.execute("PRAGMA foreign_keys = ON;")

ddl_script = '''
CREATE TABLE members (
    member_id     INTEGER PRIMARY KEY,
    nickname      TEXT NOT NULL,
    email         TEXT,
    city          TEXT,
    registered_at TEXT,
    loyalty_tier  TEXT CHECK (loyalty_tier IN ('Bronze', 'Silver', 'Gold', 'Platinum')),
    balance       REAL CHECK (balance >= 0)
);

CREATE TABLE hardware_pcs (
    pc_id       INTEGER PRIMARY KEY,
    pc_number   TEXT NOT NULL UNIQUE,
    zone_name   TEXT NOT NULL,
    hourly_rate REAL CHECK (hourly_rate > 0),
    gpu         TEXT,
    cpu         TEXT,
    ram_gb      INTEGER CHECK (ram_gb > 0),
    status      TEXT CHECK (status IN ('Active', 'Maintenance', 'Reserved'))
);

CREATE TABLE game_sessions (
    session_id     INTEGER PRIMARY KEY,
    member_id      INTEGER REFERENCES members(member_id),
    pc_id          INTEGER REFERENCES hardware_pcs(pc_id),
    game_title     TEXT NOT NULL,
    session_date   TEXT NOT NULL,
    duration_hours REAL CHECK (duration_hours > 0),
    discount_promo INTEGER CHECK (discount_promo IN (0, 1)),
    total_paid     REAL CHECK (total_paid >= 0),
    client_rating  REAL
);

CREATE TABLE bar_orders (
    order_id     INTEGER PRIMARY KEY,
    member_id    INTEGER REFERENCES members(member_id),
    item_name    TEXT NOT NULL,
    category     TEXT NOT NULL,
    order_date   TEXT NOT NULL,
    quantity     INTEGER CHECK (quantity > 0),
    item_price   REAL CHECK (item_price > 0),
    total_amount REAL CHECK (total_amount >= 0)
);

CREATE TABLE tournaments (
    tournament_id   INTEGER PRIMARY KEY,
    tournament_name TEXT NOT NULL,
    game_title      TEXT NOT NULL,
    prize_fund      REAL CHECK (prize_fund >= 0),
    entry_fee       REAL CHECK (entry_fee >= 0),
    start_date      TEXT NOT NULL,
    min_rank        TEXT
);
'''
cursor.executescript(ddl_script)

data_dir = pathlib.Path("members.csv").parent if os.path.exists("members.csv") else pathlib.Path("../variant_07")
for table in ["members", "hardware_pcs", "game_sessions", "bar_orders", "tournaments"]:
    df_temp = pd.read_csv(data_dir / f"{table}.csv", sep=";")
    df_temp.to_sql(table, conn, if_exists="append", index=False)

check_query = '''
SELECT 'members' AS tbl, COUNT(*) AS cnt FROM members
UNION ALL SELECT 'hardware_pcs', COUNT(*) FROM hardware_pcs
UNION ALL SELECT 'game_sessions', COUNT(*) FROM game_sessions
UNION ALL SELECT 'bar_orders', COUNT(*) FROM bar_orders
UNION ALL SELECT 'tournaments', COUNT(*) FROM tournaments;
'''
df_counts = pd.read_sql_query(check_query, conn)
display(df_counts)

,tbl,cnt
0,members,150
1,hardware_pcs,30
2,game_sessions,500
3,bar_orders,350
4,tournaments,8


#### 4.2. Фильтрация и сортировка данных

In [5]:
query_42 = '''
SELECT order_id, member_id, item_name, category, quantity, item_price, total_amount
FROM bar_orders
WHERE category IN ('Снеки', 'Горячее') AND total_amount >= 400
ORDER BY total_amount DESC, order_id ASC
LIMIT 7;
'''
df_res_42 = pd.read_sql_query(query_42, conn)
display(df_res_42)

,order_id,member_id,item_name,category,quantity,item_price,total_amount
0,6100,111,Бургер Классический,Горячее,4,340.0,1360.0
1,6343,139,Сэндвич с ветчиной и сыром,Горячее,4,260.0,1040.0
2,6291,93,Наггетсы куриные (8 шт),Горячее,4,250.0,1000.0
3,6304,132,Наггетсы куриные (8 шт),Горячее,4,250.0,1000.0
4,6233,46,Сэндвич с ветчиной и сыром,Горячее,3,260.0,780.0
5,6236,45,Сэндвич с ветчиной и сыром,Горячее,3,260.0,780.0
6,6237,131,Сэндвич с ветчиной и сыром,Горячее,3,260.0,780.0


#### 4.3. Группировка и фильтрация агрегированных данных

In [6]:
query_43 = '''
SELECT loyalty_tier, COUNT(*) AS members_count, ROUND(SUM(balance), 2) AS total_balance, ROUND(AVG(balance), 2) AS avg_balance
FROM members
GROUP BY loyalty_tier
HAVING SUM(balance) > 20000
ORDER BY total_balance DESC;
'''
df_res_43 = pd.read_sql_query(query_43, conn)
display(df_res_43)

,loyalty_tier,members_count,total_balance,avg_balance
0,Silver,52,131200.0,2523.08
1,Bronze,58,97600.0,1682.76
2,Gold,34,54400.0,1600.00


#### 4.4. Объединение таблиц

In [7]:
query_44 = '''
SELECT m.member_id, m.nickname, m.city, COUNT(s.session_id) AS cs_sessions, ROUND(SUM(s.total_paid), 2) AS cs_spent
FROM members m
JOIN game_sessions s ON m.member_id = s.member_id
WHERE s.game_title = 'Counter-Strike 2'
GROUP BY m.member_id, m.nickname, m.city
ORDER BY cs_spent DESC
LIMIT 10;
'''
df_res_44 = pd.read_sql_query(query_44, conn)
display(df_res_44)

,member_id,nickname,city,cs_sessions,cs_spent
0,76,Storm_351,Москва,2,1955.0
1,65,Rogue_978,Москва,2,1575.0
2,98,Turbo_572,Москва,3,1268.0
3,71,Striker_485,Екатеринбург,1,1250.0
4,12,Sniper_202,Санкт-Петербург,2,1225.0
5,38,Raven_681,Казань,1,1200.0
6,56,Dragon_882,Москва,1,1000.0
7,114,Ronin_299,Санкт-Петербург,1,950.0
8,69,Mystic_463,Екатеринбург,1,900.0
9,25,Grizzly_299,Москва,2,639.0


#### 4.5. Подзапросы

In [8]:
query_45 = '''
SELECT member_id, nickname, city, loyalty_tier, balance
FROM members
WHERE balance > (SELECT AVG(balance) FROM members WHERE loyalty_tier = 'Bronze')
ORDER BY balance DESC
LIMIT 10;
'''
df_res_45 = pd.read_sql_query(query_45, conn)
display(df_res_45)

,member_id,nickname,city,loyalty_tier,balance
0,2,Echo_117,Москва,Silver,7500.0
1,18,Matrix_285,Москва,Silver,7500.0
2,30,Sentinel_610,Москва,Silver,7500.0
3,47,Ronin_362,Санкт-Петербург,Bronze,7500.0
4,52,Storm_622,Москва,Gold,7500.0
5,67,Sniper_170,Москва,Silver,7500.0
6,85,Tempest_210,Москва,Silver,7500.0
7,91,Ghost_842,Екатеринбург,Silver,7500.0
8,94,Turbo_711,Москва,Silver,7500.0
9,103,Kraken_780,Казань,Bronze,7500.0


### Задание 5. Поиск и исправление ошибок в SQL-запросах

#### 5.1. Длительные игровые сессии на компьютерах

**Найденные ошибки:**
1. Ошибка ключа соединения: условие `ON p.pc_id = s.member_id` связывает идентификатор компьютера с ID клиента вместо `s.pc_id`.

In [9]:
query_51_fixed = '''
SELECT 
    p.pc_number,
    p.zone_name,
    s.game_title,
    s.duration_hours
FROM hardware_pcs p
JOIN game_sessions s ON p.pc_id = s.pc_id
WHERE s.duration_hours >= 3
ORDER BY s.duration_hours DESC;
'''
df_res_51 = pd.read_sql_query(query_51_fixed, conn)
display(df_res_51)

,pc_number,zone_name,game_title,duration_hours
0,PC-17,Bootcamp,Counter-Strike 2,6.0
1,PC-05,Standard,Counter-Strike 2,6.0
2,PC-17,Bootcamp,Warzone,6.0
3,PC-18,Bootcamp,Warzone,6.0
4,PC-02,Standard,Cyberpunk 2077,6.0
5,PC-19,Bootcamp,GTA V,6.0
6,PC-11,Standard,Dota 2,6.0
7,PC-17,Bootcamp,Rust,6.0
8,PC-21,Bootcamp,Cyberpunk 2077,6.0
9,PC-18,Bootcamp,Counter-Strike 2,5.0


#### 5.2. Средний чек по городам клиентов

**Найденные ошибки:**
1. Ошибка ключа соединения: `m.member_id = s.pc_id` связывает клиента с компьютером вместо `s.member_id`.
2. Агрегатная функция в `WHERE`: фильтрация по `AVG(s.total_paid) > 300` должна выполняться в секции `HAVING`.

In [10]:
query_52_fixed = '''
SELECT 
    m.city,
    COUNT(s.session_id) AS sessions_cnt,
    ROUND(AVG(s.total_paid), 2) AS avg_paid
FROM members m
JOIN game_sessions s ON m.member_id = s.member_id
GROUP BY m.city
HAVING AVG(s.total_paid) > 300
ORDER BY avg_paid DESC;
'''
df_res_52 = pd.read_sql_query(query_52_fixed, conn)
display(df_res_52)

,city,sessions_cnt,avg_paid
0,Москва,204,443.91
1,Санкт-Петербург,143,443.79
2,Екатеринбург,50,442.12
3,Казань,54,429.74
4,Новосибирск,49,427.59


#### 5.3. Сравнение выручки компьютеров со средней оплатой сессий

**Найденные ошибки:**
1. Порядок команд: блок `WHERE` размещен после `GROUP BY`, фильтрация агрегатных данных должна выполняться в `HAVING`.
2. Использование псевдонима: указан alias `avg_session_paid` вместо агрегата `AVG(s.total_paid)`.
3. Многострочный подзапрос: наличие `GROUP BY zone_name` заставляет подзапрос возвращать вектор значений вместо одного скалярного числа.

In [11]:
query_53_fixed = '''
SELECT 
    p.pc_id,
    p.pc_number,
    p.zone_name,
    ROUND(AVG(s.total_paid), 2) AS avg_session_paid
FROM hardware_pcs p
JOIN game_sessions s ON p.pc_id = s.pc_id
GROUP BY p.pc_id, p.pc_number, p.zone_name
HAVING AVG(s.total_paid) > (
    SELECT AVG(total_paid) 
    FROM game_sessions
)
ORDER BY avg_session_paid DESC
LIMIT 10;
'''
df_res_53 = pd.read_sql_query(query_53_fixed, conn)
display(df_res_53)

,pc_id,pc_number,zone_name,avg_session_paid
0,26,PC-26,VIP,915.00
1,25,PC-25,VIP,822.61
2,23,PC-23,VIP,723.36
3,24,PC-24,VIP,710.82
4,17,PC-17,Bootcamp,648.79
5,15,PC-15,Bootcamp,549.45
6,22,PC-22,Bootcamp,528.37
7,19,PC-19,Bootcamp,525.80
8,18,PC-18,Bootcamp,524.08
9,21,PC-21,Bootcamp,494.64
